# Conventional Approach — XGBoost with Optuna

Builds customer-level features from raw transaction data, tunes an XGBoost model using Optuna + cross-validation on the training set, then evaluates on a held-out test set.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from pathlib import Path
from scipy import stats
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, classification_report
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_SEED  = 42
N_TRIALS     = 100
CV_FOLDS     = 5
DATA_DIR     = Path('..') / 'data' / 'raw'
RESULTS_DIR  = Path('..') / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', None)

## 1. Build Customer-Level Features

In [ ]:
df = pd.read_csv(DATA_DIR / 'ecommerce_customer_data_large.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')
df['purchase_date']     = pd.to_datetime(df['purchase_date'])
df_sorted               = df.sort_values(['customer_id', 'purchase_date']).copy()
df_sorted['transaction_value'] = df_sorted['product_price'] * df_sorted['quantity']
DATASET_MAX_DATE        = df['purchase_date'].max()

def avg_days_between(dates):
    if len(dates) < 2:
        return np.nan
    return dates.diff().dt.days.dropna().mean()

def std_days_between(dates):
    if len(dates) < 3:
        return np.nan
    return dates.diff().dt.days.dropna().std()

def spend_trend(values):
    if len(values) < 3:
        return np.nan
    slope, *_ = stats.linregress(range(len(values)), values)
    return slope

customers = df_sorted.groupby('customer_id').agg(
    earliest_transaction_date = ('purchase_date',     'min'),
    latest_transaction_date   = ('purchase_date',     'max'),
    n_transactions            = ('purchase_date',     'count'),
    unique_product_categories = ('product_category',  'nunique'),
    avg_amount_spent          = ('transaction_value', 'mean'),
    std_amount_spent          = ('transaction_value', 'std'),
    total_spend               = ('transaction_value', 'sum'),
    total_returns             = ('returns',           'sum'),
    n_payment_methods_used    = ('payment_method',    'nunique'),
    female                    = ('gender',            lambda x: int((x == 'Female').iloc[0])),
    customer_age              = ('customer_age',      'first'),
    churn                     = ('churn',             'first'),
)

customers['days_since_last_purchase'] = (DATASET_MAX_DATE - customers['latest_transaction_date']).dt.days
customers['customer_tenure_days']     = (customers['latest_transaction_date'] - customers['earliest_transaction_date']).dt.days
customers['return_rate']              = customers['total_returns'] / customers['n_transactions']
customers['purchase_frequency']       = customers['n_transactions'] / customers['customer_tenure_days'].replace(0, np.nan)

pay_dummies = pd.get_dummies(df_sorted['payment_method'], prefix='pay')
pay_totals  = pay_dummies.join(df_sorted['customer_id']).groupby('customer_id').sum()
pay_pcts    = pay_totals.div(customers['n_transactions'], axis=0)
pay_pcts.columns = ['prct_paid_cash', 'prct_paid_credit_card', 'prct_paid_paypal']
customers   = customers.join(pay_pcts)

cat_dummies    = pd.get_dummies(df_sorted['product_category'], prefix='prct_txn')
cat_txn_totals = cat_dummies.join(df_sorted['customer_id']).groupby('customer_id').sum()
cat_txn_pcts   = cat_txn_totals.div(customers['n_transactions'], axis=0)
customers      = customers.join(cat_txn_pcts)

cat_spend = df_sorted.groupby(['customer_id', 'product_category'])['transaction_value'].sum().unstack(fill_value=0)
cat_spend.columns = [f'prct_spend_{c.lower()}' for c in cat_spend.columns]
cat_spend_pcts = cat_spend.div(customers['total_spend'], axis=0)
customers = customers.join(cat_spend_pcts).drop(columns='total_spend')

avg_time = df_sorted.groupby('customer_id')['purchase_date'].apply(avg_days_between).rename('avg_days_between_transactions')
std_time = df_sorted.groupby('customer_id')['purchase_date'].apply(std_days_between).rename('std_days_between_transactions')
trend    = df_sorted.groupby('customer_id')['transaction_value'].apply(spend_trend).rename('spend_trend')
customers = customers.join(avg_time).join(std_time).join(trend)

print(f'Customer dataset: {customers.shape}')
customers.head()

## 2. Prepare Features

In [ ]:
DROP_COLS = ['earliest_transaction_date', 'latest_transaction_date', 'churn']
X = customers.drop(columns=DROP_COLS)
y = customers['churn']

# Impute NaN for customers with too few transactions
X['std_amount_spent']              = X['std_amount_spent'].fillna(0)
X['std_days_between_transactions'] = X['std_days_between_transactions'].fillna(0)
X['spend_trend']                   = X['spend_trend'].fillna(0)
X['purchase_frequency']            = X['purchase_frequency'].fillna(X['n_transactions'])

print(f'Features : {X.shape[1]}')
print(f'Churn rate: {y.mean():.3f}')
print(f'Remaining NaN: {X.isnull().sum().sum()}')

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print(f'Train : {X_train.shape}  churn rate: {y_train.mean():.3f}')
print(f'Test  : {X_test.shape}   churn rate: {y_test.mean():.3f}')

## 4. Hyperparameter Tuning with Optuna

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

def objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 100, 1000),
        'max_depth'         : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight'  : scale_pos_weight,
        'eval_metric'       : 'auc',
        'use_label_encoder' : False,
        'random_state'      : RANDOM_SEED,
        'n_jobs'            : -1,
    }
    cv    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    aucs  = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        Xtr, Xval = X_train.iloc[train_idx], X_train.iloc[val_idx]
        ytr, yval = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = xgb.XGBClassifier(**params)
        model.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
        aucs.append(roc_auc_score(yval, model.predict_proba(Xval)[:, 1]))
    return np.mean(aucs)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest CV AUC-ROC : {study.best_value:.4f}')
print('Best params     :', study.best_params)

## 5. Train Final Model on Full Training Set

In [ ]:
best_params = study.best_params | {
    'scale_pos_weight'  : scale_pos_weight,
    'eval_metric'       : 'auc',
    'use_label_encoder' : False,
    'random_state'      : RANDOM_SEED,
    'n_jobs'            : -1,
}

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)
print('Final model trained.')

## 6. Evaluate on Held-Out Test Set

In [ ]:
y_pred_proba = final_model.predict_proba(X_test)[:, 1]
y_pred       = final_model.predict(X_test)

metrics = {
    'auc_roc'   : round(roc_auc_score(y_test, y_pred_proba), 4),
    'f1'        : round(f1_score(y_test, y_pred), 4),
    'precision' : round(precision_score(y_test, y_pred), 4),
    'recall'    : round(recall_score(y_test, y_pred), 4),
}

print('=== Test Set Results ===')
for k, v in metrics.items():
    print(f'  {k:<12}: {v}')
print('\n', classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

## 7. Save Results

In [ ]:
output = {
    'approach'    : 'conventional_xgboost',
    'metrics'     : metrics,
    'best_params' : study.best_params,
    'cv_auc'      : round(study.best_value, 4),
    'n_trials'    : N_TRIALS,
    'cv_folds'    : CV_FOLDS,
    'train_size'  : len(X_train),
    'test_size'   : len(X_test),
    'n_features'  : X_train.shape[1],
}

out_path = RESULTS_DIR / 'conventional_xgboost.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Results saved to {out_path}')